In [1]:
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import matplotlib.pyplot as plt

# NLP Resources
max_text_len = 100
max_words = 10000

In [2]:
def build_multimodal_model():
    # --- 1. TEXT BRANCH (NLP) ---
    text_input = layers.Input(shape=(max_text_len,), name="text_input")
    x_text = layers.Embedding(max_words, 64)(text_input)
    x_text = layers.Bidirectional(layers.LSTM(64))(x_text)
    x_text = layers.Dense(64, activation='relu')(x_text)

    # --- 2. IMAGE BRANCH (Computer Vision) ---
    image_input = layers.Input(shape=(224, 224, 3), name="image_input")
    # Using MobileNetV2 as a pretrained feature extractor
    base_model = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3),
                                                   include_top=False,
                                                   weights='imagenet')
    base_model.trainable = False # Freeze weights
    x_img = base_model(image_input)
    x_img = layers.GlobalAveragePooling2D()(x_img)
    x_img = layers.Dense(64, activation='relu')(x_img)

    # --- 3. MULTIMODAL FUSION ---
    # Concatenate text features and image features
    merged = layers.Concatenate()([x_text, x_img])

    # Dense layers to learn correlations between text and image
    x = layers.Dense(128, activation='relu')(merged)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)

    output = layers.Dense(1, activation='sigmoid', name="prediction")(x)

    # Final Model
    model = Model(inputs=[text_input, image_input], outputs=output)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    return model

multimodal_model = build_multimodal_model()
multimodal_model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text_input          │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ image_input         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 100, 64)   │    640,000 │ text_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mobilenetv2_1.00_2… │ (None, 7, 7,      │  2,257,984 │ image_input[0][0] │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 128)       │     66,048 │ embedding[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ mobilenetv2_1.00… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      8,256 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │     81,984 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 128)       │          0 │ dense[0][0],      │
│ (Concatenate)       │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │     16,512 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64)        │      8,256 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ prediction (Dense)  │ (None, 1)         │         65 │ dense_3[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,079,105 (11.75 MB)

 Trainable params: 821,121 (3.13 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [3]:
def generate_dummy_data(num_samples=100):
    # Random text sequences
    X_text = np.random.randint(0, max_words, size=(num_samples, max_text_len))
    # Random images (224x224 RGB)
    X_img = np.random.rand(num_samples, 224, 224, 3).astype(np.float32)
    # Random labels (0 or 1)
    Y = np.random.randint(0, 2, size=(num_samples, 1))
    return X_text, X_img, Y

text_train, img_train, y_train = generate_dummy_data(50)

print("Training on Multimodal Data...")
multimodal_model.fit(
    {"text_input": text_train, "image_input": img_train},
    y_train,
    epochs=5,
    batch_size=8
)

Training on Multimodal Data...
Epoch 1/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 12s 395ms/step - accuracy: 0.6400 - loss: 0.6936
Epoch 2/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 4s 279ms/step - accuracy: 0.4800 - loss: 0.7228
Epoch 3/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 266ms/step - accuracy: 0.6200 - loss: 0.7913
Epoch 4/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 269ms/step - accuracy: 0.6200 - loss: 0.6328
Epoch 5/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 411ms/step - accuracy: 0.9000 - loss: 0.5159


In [4]:
def explainable_prediction(text_str, image_array):
    # 1. Preprocess inputs (Simulated)
    # In a real app, you'd use tokenizer.texts_to_sequences here
    text_processed = np.random.randint(0, max_words, size=(1, max_text_len))
    img_processed = np.expand_dims(image_array, axis=0)

    # 2. Get Score
    score = multimodal_model.predict({"text_input": text_processed, "image_input": img_processed}, verbose=0)[0][0]

    # 3. Logic-based Explanation
    label = "FAKE" if score > 0.5 else "REAL"
    confidence = score if score > 0.5 else 1 - score

    print(f"Result: {label} ({confidence*100:.2f}% confidence)")
    print("\n--- System Explanation ---")
    if score > 0.8:
        print("[!] High Alert: Text sentiment and Image metadata show strong signs of manipulation.")
    elif score > 0.5:
        print("[?] Warning: Minor mismatch detected between the headline and the visual context.")
    else:
        print("[✓] Verified: Text and Image contents are consistent with credible sourcing.")

# Test with a blank image
test_img = np.random.rand(224, 224, 3)
explainable_prediction("Breaking: Scientists discover gold on Mars!", test_img)

Result: REAL (51.89% confidence)

--- System Explanation ---
[✓] Verified: Text and Image contents are consistent with credible sourcing.
